# Deduplicate cyteonto CSVs

If cyteonto is run on cell level, you get tons of lines per CSV. They need to be deduplicated and can then be stacked into one CSV, which gets cached in [`output/cyteonto_pipeline`](../../output/cyteonto_pipeline/).

In [ ]:
import os
from pathlib import Path
import datetime

import pandas as pd

from shared.repo import REPO_ROOT
from cyteonto import dedup_table

In [ ]:
dirs: list[Path] = [
    REPO_ROOT / "output/cyteonto_pipeline/20260526_112224/results",
    REPO_ROOT / "output/cyteonto_pipeline/20260526_121155/results",
]

csv_list: list[Path] = []
for dir in dirs:
    file_names: list[str] = os.listdir(dir)
    file_paths: list[Path] = []
    for file in file_names:
        file_paths.append(dir / file)

    csv_list.extend(file_paths)

In [ ]:
OUTPUT_CSV_STEM = "deduplicated"

stacked_df_exists = False
for i, csv in enumerate(csv_list):
    accession = csv.name.split("_")[0]
    df = dedup_table(csv, accession)
    if df is not None:
        if not stacked_df_exists:
            stacked_df = df
            stacked_df_exists = True
        else:
            stacked_df = pd.concat([stacked_df, df])

stacked_csv_dir = REPO_ROOT / "output/cyteonto_pipeline/deduplicated_tables/"
os.makedirs(stacked_csv_dir, exist_ok=True)

stacked_df.to_csv(stacked_csv_dir / (OUTPUT_CSV_STEM + ".csv"), index=False)